In [0]:
%run "/Workspace/Users/jeevan.azureacc3@gmail.com/bankaml-de-project/notebooks/04_utils/watermark_incremental_load"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
transactions_bronze_df = spark.read.table("bankaml.bronze.transactions")
transactions_bronze_df = get_new_rows_from_source_df(transactions_bronze_df, "silver", "silver", "transactions")

quarantine_df = transactions_bronze_df.filter(
    (col("txn_id").isNull()) | 
    (col("account_id").isNull()) | 
    (col("amount").cast('double') <= 0)
)
transactions_bronze_df = transactions_bronze_df.filter(
    (col("txn_id").isNotNull()) &
    (col("account_id").isNotNull()) &
    (col("amount").cast('double') > 0)
)
accounts_table_df = DeltaTable.forName(spark, "bankaml.silver.accounts").toDF()
txn_accts_joined_df = transactions_bronze_df.alias("s").join(
    accounts_table_df.alias("t"), 
    col("s.account_id") == col("t.account_id"), 
    "left"
)
transactions_bronze_df = txn_accts_joined_df.filter(col("t.account_id").isNotNull()).select("s.*")

quarantine_df_2 = txn_accts_joined_df.filter(col("t.account_id").isNull()).select("s.*").withColumn("quarantine_reason", lit("account_id is not exists"))

In [0]:
transactions_bronze_df = transactions_bronze_df.dropDuplicates(["txn_id"])
transactions_df = (
    transactions_bronze_df.withColumn("amount", col("amount").cast(DoubleType())) 
        .withColumn("created_at", col("created_at").cast(TimestampType())) 
        .withColumn("currency", upper(col("currency")))
        .withColumn("counterparty_acct", coalesce(col("counterparty_acct"), lit("CASH")))
        .withColumn("channel", coalesce(col("channel"), lit("unknown")))
        .withColumn("txn_ts", 
                    when(col("txn_ts").contains("T"), 
                         to_utc_timestamp(
                             col("txn_ts").cast(TimestampType()),
                              "UTC"
                        )
                    )
                    .otherwise(
                        to_utc_timestamp(
                            to_timestamp(col("txn_ts"), "yyyy-MM-dd HH:mm:ss"),
                            "Asia/Kolkata"
                        )
                    )
        ) 
)

fx_rates_df = spark.read.table("bankaml.silver.fx_rates")
transactions_df = (
    transactions_df.alias("s").join(
        fx_rates_df.alias("t"),
        col("s.currency") == col("t.currency"),
        "left"
    )
).select("s.*", round(col("s.amount") * col("t.rate_to_usd"), 2).alias("amount_usd"))

transactions_df = transactions_df.withColumnRenamed("source_system", "created_by")

In [0]:
%sql
create table if not exists bankaml.silver.transactions
(
    txn_id string,
    account_id string,
    txn_ts timestamp,
    txn_type string,
    amount decimal(18,2),
    currency string,
    amount_usd decimal(18,2),
    counterparty_acct string,
    channel string,
    created_at timestamp,
    created_by string,
    _ingestion_ts timestamp
)
using delta;

In [0]:
transaction_target_table = DeltaTable.forName(spark, "bankaml.silver.transactions")
transactions_df = transactions_df.select(*transaction_target_table.toDF().columns)

(
    transaction_target_table.alias("t").merge(
        transactions_df.alias("s"),
        "t.txn_id = s.txn_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

update_last_processed_value(transactions_bronze_df, "silver", "silver", "transactions")

In [0]:
quarantine_df = quarantine_df.withColumns(
    {
        "quarantine_reason": 
            when(col("txn_id").isNull() | col("account_id"), lit("txn_id/account_id is null"))
            .when(col("amount").cast('double') <= 0, lit("invalid amount"))
            .otherwise(lit("unknown"))
        ,
        "quarantined_at": current_timestamp(),
        "source_layer": lit("silver")
    }
)

quarantine_df_2 = quarantine_df_2.withColumns(
    {
    "quarantined_at": current_timestamp(),
        "source_layer": lit("silver")
    }
)
quarantine_df = quarantine_df.unionByName(quarantine_df_2)

In [0]:
%sql
create table if not exists bankaml.quarantine.transactions
(
    txn_id string,
    account_id string,
    txn_ts string,
    txn_type string,
    amount string,
    currency string, 
    counterparty_acct string,
    channel string,
    created_at string,
    source_system string,
    _ingestion_ts TIMESTAMP,
    quarantine_reason STRING,  
    quarantined_at TIMESTAMP,
    source_layer STRING
)
using delta;

In [0]:
quarantine_df.write.format("delta").mode("append").saveAsTable("bankaml.quarantine.transactions")